# Paper positions: interactive piece interactions

The positions from the chessplainer paper (`chessplainer.POSITIONS`), each explained with Shapley
interactions (see `example_interactions.ipynb`).

**The viewers are interactive**: click a piece to see all its interactions with the other pieces
(partners colored by the value of their pair, arrows to the strongest ones), click a row of the table
to jump to that partner, click an empty square to go back to the overview. They are plain HTML: they
keep working in exported or saved notebooks, and `viewer.save("page.html")` writes a standalone page.

Requires `pip install "chessplainer[shapiq]"` (Python >= 3.12) and Stockfish.

In [ ]:
import chess
import pandas as pd

from chessplainer import POSITIONS, ChessExplainer, EngineWrapper, board_svg

In [ ]:
def explain_position(name, depth=12, **kwargs):
    """Explains a paper position with k-SII interactions (exact up to 14 pieces)."""
    board = chess.Board(POSITIONS[name]["fen"])
    with EngineWrapper(path="stockfish", predict_limit_kwargs=dict(depth=depth)) as engine:
        engine.fit([board])
        explainer = ChessExplainer(board, engine)
        explainer.explain_interactions(index="k-SII", max_order=2, **kwargs)
    return explainer


overview = pd.DataFrame(
    [
        {
            "name": name,
            "title": position["title"],
            "pieces": len(chess.Board(position["fen"]).piece_map()) - 2,
            "to move": "white" if chess.Board(position["fen"]).turn else "black",
            "fen": position["fen"],
        }
        for name, position in POSITIONS.items()
    ]
).sort_values("pieces")
overview

## Exact interactions

With up to 14 pieces (kings excluded) every subset of pieces is evaluated, so the values are exact.
The positions with the most pieces here take a few minutes each at depth 12.

### Two rooks against a queen

`example`: 3 pieces, White to move, 8 evaluations.

In [ ]:
explainer = explain_position("example")
explainer.interactions_viewer()

### Tactic 2

`tactic_2`: 4 pieces, White to move, 16 evaluations.

In [ ]:
explainer = explain_position("tactic_2")
explainer.interactions_viewer()

### Tactic: mate in 2

`tactic_1`: 5 pieces, White to move, 32 evaluations.
Source: https://www.reddit.com/r/ChessPuzzles/comments/1lf799h/white_to_play_and_mate_in_2/

In [ ]:
explainer = explain_position("tactic_1")
explainer.interactions_viewer()

### Knight against bishop endgame (draw)

`knight_vs_bishop_endgame_draw`: 6 pieces, White to move, 64 evaluations.

In [ ]:
explainer = explain_position("knight_vs_bishop_endgame_draw")
explainer.interactions_viewer()

### Good knight against bad bishop

`good_knight_bad_bishop`: 8 pieces, Black to move, 256 evaluations.
Source: https://en.chessbase.com/post/endgame-blog-karsten-mueller-2020_38e

In [ ]:
explainer = explain_position("good_knight_bad_bishop")
explainer.interactions_viewer()

### Self-blocking pawn

`self_blocking_pawn`: 9 pieces, White to move, 512 evaluations.

In [ ]:
explainer = explain_position("self_blocking_pawn")
explainer.interactions_viewer()

### Trapped rook

`trapped_rook`: 10 pieces, White to move, 1024 evaluations.

In [ ]:
explainer = explain_position("trapped_rook")
explainer.interactions_viewer()

### Knight against bishop endgame puzzle

`knight_vs_bishop_endgame`: 10 pieces, White to move, 1024 evaluations.

In [ ]:
explainer = explain_position("knight_vs_bishop_endgame")
explainer.interactions_viewer()

### Active king (Khismatullin vs Eljanov)

`active_king_2`: 14 pieces, White to move, 16384 evaluations.
Source: https://www.chess.com/article/view/the-craziest-king-moves-ever

In [ ]:
explainer = explain_position("active_king_2")
explainer.interactions_viewer()

## Approximated interactions

With more pieces the values are approximated (32768 evaluations by default): a shallower search
(depth 8) keeps each position to a few minutes. Pairwise values carry some sampling noise, and in
crowded positions the strongest ones are mostly about material: see `example_interactions.ipynb`.

### Engines disagree (Stockfish and Lc0)

`engines`: 16 pieces, White to move.
Source: https://lichess.org/study/wxrovYNH/hBLBToEN

In [ ]:
explainer = explain_position("engines", depth=8, random_state=0)
explainer.interactions_viewer()

### Active king

`active_king`: 22 pieces, White to move.
Source: https://www.chess.com/forum/view/general/king-as-offensive-piece

In [ ]:
explainer = explain_position("active_king", depth=8, random_state=0)
explainer.interactions_viewer()

### Too many pieces (starting position)

`too_many_pieces`: 30 pieces, White to move.

In [ ]:
explainer = explain_position("too_many_pieces", depth=8, random_state=0)
explainer.interactions_viewer()